# Per-motif inference accuracy

Driver notebook for `motif_scoring.py` (pure, dataset-agnostic detection + scoring
functions live there, not here). Two parts:

1. **Detect** which gene-triples in a chosen ground-truth network belong to each of
   4 motif types: `fan_out`, `feed_forward`, `regulated_mutual`, `auto_regulation`
   (external regulators of a self-looping target).
2. **Score** every BEELINE method's predictions specifically on each motif's
   defining edges, averaged across all available replicate runs.

**To point this at a different dataset**, edit only the Config cell below —
everything after it is dataset-agnostic. Three example configs are given
(commented out except the active one): a `real_networks` dataset (B_cell_activation /
Circadian_cycle, this session's own output), a curated real-world network (GSD/HSC/
VSC/mCAD, `network_sweep_analysis.ipynb`'s track), and the synthetic `network_sweep`
topology family.

In [ ]:
import glob, json, os, re
import pandas as pd

from benchmarks.network_benchmarks.score.motif_scoring import (
    load_ground_truth, detect_motifs,
    predicted_edges_from_ranked, predicted_signed_edges_from_ranked,
    twinfer_predicted_edges,
    score_methods_by_motif,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Config — pick a dataset

`GT_PATH` must be TwINFER's native signed-adjacency `.txt` format (headerless,
comma-separated). `PREDICTIONS_ROOT` is the BEELINE-convention dataset directory
containing `simrep*_{SCHEME}/<ALGORITHM>/rankedEdges.csv` run subdirectories — every
replicate found there is scored and averaged.

In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
REPO = f'{TWINFER_PROJECT_ROOT}'

# --- Example 1 (active): real_networks, B_cell_activation (10 genes) ---
# GT_PATH = f'{REPO}/input_data/real_world_networks/B_cell.txt'
# PREDICTIONS_ROOT = f'{REPO}/analysis_data/paper_analysis/real_networks/beeline_inference/B_cell_activation'
# SCHEME = 'twin_paired'   # or 'spread'

# --- Example 2: real_networks, Circadian_cycle (4 genes, has a self-loop) ---
# GT_PATH = f'{REPO}/input_data/real_world_networks/circadian.txt'
# PREDICTIONS_ROOT = f'{REPO}/analysis_data/paper_analysis/real_networks/beeline_inference/Circadian_cycle'
# SCHEME = 'twin_paired'

# --- Example 3: curated real-world network (GSD/HSC/VSC/mCAD), TwINFER-sim track ---
GT_PATH = f'{REPO}/input_data/real_world_networks/GSD.txt'
# [2026-10-01 commented out: scratch purged; same data in benchmarking_analysis/]
# PREDICTIONS_ROOT = '/scratch/gzu5140/twinfer_real/outputs/GSD'
PREDICTIONS_ROOT = f'{TWINFER_PROJECT_ROOT}/benchmarking_analysis/twinfer_real/outputs/GSD'
SCHEME = 'twin_paired'

# --- Example 4: synthetic network_sweep topology (one grn_n6_* family member) ---
# GT_PATH = f'{REPO}/input_data/network_sweep_final/grn_n6_e9_pos50_sign_ratio_rep1.txt'
# PREDICTIONS_ROOT = '/home/gzu5140/TwINFER_KA/analysis_data/synthetic_network_benchmark_06082026/beeline_inference/grn_n6_e9_pos50_sign_ratio_rep1'
# SCHEME = 'spread'

# Top-k budget for turning each method's ranking into a hard-decision edge set.
# None -> use the ground truth's own total nonzero-edge count (a common, neutral
# default matching the top-k convention used throughout this project's notebooks).
TOP_K = None

# TwINFER's own predictions come from a fanout_on/fanout_off all_results.json pair
# (not rankedEdges.csv) -- see benchmark_twinfer_vs_boolode.ipynb for the same
# convention. This root + regime (z>8, eliminate false arms) is for TwINFER's own
# Gillespie-simulated datasets (all 4 examples above); a BoolODE-simulated dataset
# would need a different JSON source and the z<137/flipped/no-eliminate regime --
# see benchmark_twinfer_vs_boolode.ipynb's 6b if you need that track here too.
#
# The JSON filenames are keyed by the dataset/folder name (PREDICTIONS_ROOT's
# basename), auto-derived below -- NOT necessarily the "type" token embedded in
# the raw simulation CSV filenames, which can differ (e.g. Circadian_cycle's raw
# sims use "Circadian", but infer_real_networks_extra.py's JSON output for it is
# keyed "Circadian_cycle", matching the folder name). If TwINFER's predictions
# aren't found for a network whose JSON really does use a different key, override
# TWINFER_NET_TOKEN explicitly below instead of relying on the auto-derived one.
# [2026-10-01 commented out: scratch purged; same data in benchmarking_analysis/]
# TWINFER_JSON_ROOT = '/scratch/gzu5140/twinfer_nb'
TWINFER_JSON_ROOT = f'{TWINFER_PROJECT_ROOT}/benchmarking_analysis/twinfer_nb'
TWINFER_THRESHOLD = 8.0
TWINFER_FLIPPED = False
TWINFER_ELIMINATE_FALSE_ARMS = True
TWINFER_NET_TOKEN = None   # None -> auto-derive from PREDICTIONS_ROOT's basename

## Part 1 — detect motifs

In [ ]:
gt = load_ground_truth(GT_PATH)
print(f'{len(gt)} genes')

motifs = detect_motifs(gt)
summary = pd.DataFrame([
    {'motif': name, 'n_instances': len(insts),
     'n_defining_edges': sum(len(i['edges']) for i in insts)}
    for name, insts in motifs.items()
])
summary

## Part 2 — score every method, averaged across replicates

Every `simrep*_{SCHEME}` run directory found under `PREDICTIONS_ROOT` is scored
independently (own top-k budget from the same `TOP_K`/ground-truth edge count), then
averaged per (method, motif).

In [ ]:
k = TOP_K if TOP_K is not None else int((gt.to_numpy() != 0).sum())
print('top-k budget:', k)

twinfer_net_token = TWINFER_NET_TOKEN or os.path.basename(PREDICTIONS_ROOT.rstrip('/'))
print('TwINFER JSON key:', twinfer_net_token)


def find_twinfer_json(json_dir, net_token, rep_suffix):
    """Locate {net_token}_rep_..._{rep_suffix}_all_results.json under json_dir.
    Filename convention isn't uniform across when a network's JSON was generated:
    newer runs (B_cell_activation, Circadian_cycle) use "{token}_rep_{N}", older
    ones (GSD/HSC/VSC/mCAD) use "{token}_rep_{config_idx}_{rep_idx}_{8charhash}"
    -- extra numeric segments before the trailing identifier. In both cases the
    LAST underscore-separated token before "_all_results.json" is exactly the
    same rep identifier the BEELINE run directory uses (simrep<rep_suffix>_...),
    so match on that rather than assuming a fixed filename shape."""
    for path in glob.glob(f'{json_dir}/{net_token}_rep_*_all_results.json'):
        stem = os.path.basename(path)[: -len('_all_results.json')]
        if stem.split('_')[-1] == rep_suffix:
            return path
    return None


RUN_ID_RE = re.compile(r'^simrep(.+)_(?:spread|twin_paired)$')
run_dirs = sorted(glob.glob(f'{PREDICTIONS_ROOT}/simrep*_{SCHEME}'))
print(f'{len(run_dirs)} replicate run(s) found')

per_rep_frames = []
n_twinfer_found = 0
for run_dir in run_dirs:
    rep_id = os.path.basename(run_dir)
    method_predictions, method_predictions_signed = {}, {}

    for algo_dir in sorted(glob.glob(f'{run_dir}/*')):
        ranked_path = f'{algo_dir}/rankedEdges.csv'
        if not os.path.exists(ranked_path):
            continue
        algo = os.path.basename(algo_dir)
        ranked = pd.read_csv(ranked_path, sep=None, engine='python')
        method_predictions[algo] = predicted_edges_from_ranked(ranked, k)
        method_predictions_signed[algo] = predicted_signed_edges_from_ranked(ranked, k)

    # TwINFER's own predictions -- separate JSON source (fanout_on/fanout_off
    # all_results.json pair), matched to this same replicate by its rep suffix.
    m = RUN_ID_RE.match(rep_id)
    if m:
        rep_suffix = m.group(1)
        fon = find_twinfer_json(f'{TWINFER_JSON_ROOT}/fanout_on', twinfer_net_token, rep_suffix)
        foff = find_twinfer_json(f'{TWINFER_JSON_ROOT}/fanout_off', twinfer_net_token, rep_suffix)
        if fon and foff:
            with open(fon) as f:
                don = json.load(f)
            with open(foff) as f:
                doff = json.load(f)
            edges, edges_signed = twinfer_predicted_edges(
                don, doff, threshold=TWINFER_THRESHOLD, flipped=TWINFER_FLIPPED,
                eliminate_false_arms=TWINFER_ELIMINATE_FALSE_ARMS, k=k,
            )
            method_predictions['TwINFER'] = edges
            method_predictions_signed['TwINFER'] = edges_signed
            n_twinfer_found += 1

    if not method_predictions:
        continue
    rep_result = score_methods_by_motif(motifs, method_predictions, method_predictions_signed)
    rep_result.insert(0, 'rep', rep_id)
    per_rep_frames.append(rep_result)

if n_twinfer_found == 0:
    print(f"WARNING: no TwINFER JSON found under {TWINFER_JSON_ROOT}/fanout_on|off/ for key "
          f"'{twinfer_net_token}' -- TwINFER will be missing from the results below. "
          f"Either it hasn't been generated yet for this dataset (see "
          f"infer_real_networks_extra.py), or TWINFER_NET_TOKEN needs to be set explicitly "
          f"in the Config cell (the auto-derived key doesn't always match -- see the comment there).")
elif n_twinfer_found < len(run_dirs):
    print(f'NOTE: TwINFER found for {n_twinfer_found}/{len(run_dirs)} replicates only.')

all_reps = pd.concat(per_rep_frames, ignore_index=True)
print(f'{all_reps["rep"].nunique()} replicate(s) x {all_reps["method"].nunique()} method(s) scored:',
      sorted(all_reps['method'].unique()))
all_reps.head()

In [ ]:
# Mean +/- across replicates, one row per (method, motif)
_agg_cols = [c for c in ['recall', 'recall_signed', 'confound_fp_rate'] if c in all_reps.columns]
mean_by_motif = (
    all_reps.groupby(['motif', 'method'])[_agg_cols]
    .mean()
    .round(4)
)
for motif_name in motifs:
    print(f'\n=== {motif_name} ===')
    sub = mean_by_motif.loc[motif_name].dropna(axis=1, how='all')
    if sub.empty:
        print('  (no instances of this motif in the chosen network)')
        continue
    display(sub.sort_values(sub.columns[0], ascending=False))

## Things to try

* Switch `SCHEME` between `'spread'` and `'twin_paired'` to compare BEELINE sampling schemes.
* Point `PREDICTIONS_ROOT`/`GT_PATH` at any other dataset with the same
  `simrep*_{scheme}/<ALGORITHM>/rankedEdges.csv` layout — nothing below the Config
  cell needs to change.
* Extend `motif_scoring.py` with a 5th motif type by adding one more `detect_*`
  function and one entry in `detect_motifs()`'s dispatch dict — this notebook picks
  it up automatically (the Part 1/2 loops iterate over whatever `detect_motifs()`
  returns, not a hardcoded list).